# LangGraph — Intermediate Agent

**Framework:** LangGraph  
**Level:** 02 — Intermediate  
**Model:** `gemini-2.0-flash`

### What's New vs Simple
| Feature | Simple | Intermediate |
|---|---|---|
| State fields | Only `messages` | **Richer state**: `cities_researched`, `last_city` |
| Persistence | None (in-process) | **`MemorySaver` checkpointer** (survives across `.invoke()` calls) |
| Tools | 2 | **3** (added `get_travel_advisory`) |
| Session | Stateless | **`thread_id`** routes to persistent state |

## Concept: MemorySaver + Richer State

```
graph.invoke(input, config={"thread_id": "abc"})
                      │
                      ▼
┌───────────────────────────────────────┐
│           MemorySaver                 │
│   Loads checkpoint for thread "abc"   │
│                                       │
│   PlannerState {                      │
│     messages: [...Turn 1, Turn 2...]  │  ← persisted messages
│     cities_researched: ["Tokyo"]      │  ← persisted custom field
│     last_city: "Tokyo"                │  ← persisted custom field
│   }                                   │
└────────────┬──────────────────────────┘
             │ restore + run graph
             ▼
       [llm_node] → [tools_node] → [llm_node]
             │
             ▼
┌───────────────────────────────────────┐
│           MemorySaver                 │
│   Saves updated checkpoint            │
│   cities_researched: ["Tokyo","Bgl"]  │
└───────────────────────────────────────┘
```

**Two LangGraph-specific insights:**
1. **State is more than messages** — you can track any structured data alongside the conversation
2. **MemorySaver** serializes the entire state between runs — so the graph can be stopped and resumed

## Setup

In [ ]:
import os
from typing import Annotated, Optional
from typing_extensions import TypedDict
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env"
print("✓ Ready")

## 3 Tools

In [ ]:
@tool
def get_weather(city: str) -> dict:
    """Return mock weather data for a city."""
    data = {
        "london":    {"condition": "Cloudy",       "temp_c": 12},
        "new york":  {"condition": "Sunny",         "temp_c": 22},
        "bangalore": {"condition": "Rainy",         "temp_c": 25},
        "tokyo":     {"condition": "Clear",         "temp_c": 18},
        "paris":     {"condition": "Partly Cloudy", "temp_c": 16},
    }
    key = city.lower()
    return {"city": city, **data[key]} if key in data else {"error": f"No data for '{city}'."}


@tool
def get_time(city: str) -> dict:
    """Return current local time for a city."""
    times = {
        "london": "14:30 GMT", "new york": "09:30 EST",
        "bangalore": "20:00 IST", "tokyo": "22:30 JST", "paris": "15:30 CET",
    }
    key = city.lower()
    return {"city": city, "local_time": times[key]} if key in times else {"error": f"No time data."}


@tool
def get_travel_advisory(city: str) -> dict:  # ← NEW
    """Return travel safety advisory for a city."""
    advisories = {
        "london":    {"level": "Low",    "notes": "Routine precautions."},
        "new york":  {"level": "Low",    "notes": "Normal precautions."},
        "bangalore": {"level": "Medium", "notes": "Monsoon affects transport."},
        "tokyo":     {"level": "Low",    "notes": "Very safe city."},
        "paris":     {"level": "Low",    "notes": "Alert in crowded spots."},
    }
    key = city.lower()
    if key in advisories:
        a = advisories[key]
        return {"city": city, "advisory_level": a["level"], "notes": a["notes"]}
    return {"error": f"No advisory data for '{city}'."}


tools = [get_weather, get_time, get_travel_advisory]
print(f"{len(tools)} tools defined")

## Richer State Definition

At the Intermediate level, state holds **more than just messages**. Custom fields are updated by nodes and persist across turns via `MemorySaver`.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # full conversation history
    cities_researched: list          # tracks which cities were covered this session
    last_city: Optional[str]         # last city mentioned — for context in later turns

print("State fields:", list(AgentState.__annotations__.keys()))

## Nodes + Graph with MemorySaver

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_MSG = SystemMessage(content=(
    "You are a professional travel assistant. "
    "When asked about a city, always call get_weather, get_time, AND get_travel_advisory. "
    "Remember context from earlier in this conversation."
))

KNOWN_CITIES = ["london", "new york", "bangalore", "tokyo", "paris"]

def llm_node(state: AgentState) -> dict:
    messages = [SYSTEM_MSG] + state["messages"]
    response = llm_with_tools.invoke(messages)

    # Update custom state fields based on the latest user message
    cities = list(state.get("cities_researched") or [])
    last_city = state.get("last_city")
    if state["messages"]:
        last_human = state["messages"][-1].content
        for c in KNOWN_CITIES:
            if c in last_human.lower() and c.title() not in cities:
                cities.append(c.title())
                last_city = c.title()

    return {"messages": [response], "cities_researched": cities, "last_city": last_city}


tools_node = ToolNode(tools)

builder = StateGraph(AgentState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tools_node)
builder.set_entry_point("llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

# MemorySaver: persists state between .invoke() calls for the same thread_id
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled with MemorySaver")

## Multi-Turn with thread_id

In [ ]:
def run(query: str, thread_id: str = "thread_01") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=config,
    )
    return result["messages"][-1].content


THREAD = "travel_thread"

# Turn 1
q1 = "Give me a full travel briefing for Tokyo."
print(f"Turn 1 — {q1}")
print(f"Agent: {run(q1, THREAD)}\n")

In [ ]:
# Turn 2
q2 = "Now do the same for Bangalore. I prefer warm weather — which would you recommend?"
print(f"Turn 2 — {q2}")
print(f"Agent: {run(q2, THREAD)}\n")

In [ ]:
# Turn 3
q3 = "Based on what you told me, what's the safer city to visit?"
print(f"Turn 3 — {q3}")
print(f"Agent: {run(q3, THREAD)}\n")

In [ ]:
# Inspect persisted state — what MemorySaver knows about this thread
config = {"configurable": {"thread_id": THREAD}}
snapshot = graph.get_state(config)

print("State snapshot:")
print(f"  cities_researched : {snapshot.values.get('cities_researched', [])}")
print(f"  last_city         : {snapshot.values.get('last_city')}")
print(f"  total messages    : {len(snapshot.values.get('messages', []))}")

In [ ]:
# Fresh thread = blank state
q_new = "What did you tell me about Tokyo?"
print(f"Fresh thread — {q_new}")
print(f"Agent: {run(q_new, 'fresh_thread')}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `cities_researched` in state | State can track structured domain data — not just messages |
| `MemorySaver` checkpointer | Serializes entire state; graph can be paused + resumed |
| `thread_id` in config | Routes each call to its persistent checkpoint |
| `graph.get_state(config)` | Inspect state at any point — great for debugging |
| Fresh thread = blank state | Each `thread_id` is fully isolated |

### Why MemorySaver is Powerful
In production, swap `MemorySaver` for `SqliteSaver` or `PostgresSaver` — the same API, but state survives server restarts, making long-running autonomous agents possible.

### Next: [03-Complex →](../03-complex/agent.ipynb)